# Medical Report Generation System

Time estimate: **45 minutes**

## Objectives

After completing this lab, you will be able to:
- Generate structured radiology reports from imaging findings and patient data
- Process and analyze medical imaging metadata and clinical observations
- Build a rule-based and template-driven report generation system
- Apply natural language processing techniques to medical text generation
- Evaluate the quality and consistency of automated medical reports

## What you will do

In this lab, you will build an automated system that generates structured radiology reports based on imaging findings and patient history. You'll learn to process clinical data, apply medical templates, and create consistent, professional medical documentation that can assist radiologists in their workflow.

## Overview

Radiology reports are critical documents that communicate imaging findings to referring physicians and guide clinical decision-making. However, generating these reports is time-consuming and requires careful attention to detail, standardized terminology, and structured formatting. Radiologists often spend 30-50% of their time on report generation, which can lead to fatigue and potential inconsistencies.

Automated report generation systems can significantly improve workflow efficiency by creating initial draft reports from structured findings. These systems help standardize terminology, reduce documentation time, and allow radiologists to focus more on image interpretation rather than typing. Modern AI-assisted reporting tools are increasingly used in clinical practice to enhance productivity while maintaining quality.

In this lab, you'll build a system that takes imaging findings (such as presence of fractures, lesions, or normal anatomy) along with patient demographics and clinical history, and generates properly formatted radiology reports. This approach mirrors real-world clinical decision support systems used in hospitals today.

## About the Dataset

For this lab, you will use a **synthetic radiology dataset** created specifically for educational purposes. The dataset simulates chest X-ray examinations with various findings including normal studies, pneumonia, fractures, pleural effusions, and other common radiological findings.

**Input Features:**
- Patient demographics (age, gender, patient ID)
- Clinical indication (reason for examination)
- Imaging modality (e.g., Chest X-ray, CT Chest)
- Anatomical findings (lung fields, heart size, bones, pleura)
- Pathological findings (presence/absence of abnormalities)
- Severity indicators (mild, moderate, severe)

**Output:**
- Structured radiology report with standard sections: Clinical History, Technique, Findings, and Impression

Note: This is a simulated dataset for educational use only and does not contain real patient information.

## Setup

In [ ]:
# Install required packages for data manipulation, visualization, and NLP
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Import libraries for data manipulation
import pandas as pd
import numpy as np

# Import libraries for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import libraries for text processing and date handling
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configure visualization settings for better appearance
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display all columns in pandas dataframes
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [ ]:
# Print library versions for reproducibility
import sys
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

## Step 1: Generate synthetic radiology dataset

You'll create a synthetic dataset that simulates radiology examination data with various findings. This includes patient information, examination details, and structured findings that will be used to generate reports.

In [ ]:
# Define possible values for each clinical parameter

# Patient demographics options
genders = ['Male', 'Female']

# Clinical indications (reasons for examination)
indications = [
    'Shortness of breath',
    'Chest pain',
    'Cough and fever',
    'Follow-up study',
    'Post-operative evaluation',
    'Trauma',
    'Routine screening'
]

# Imaging modalities
modalities = ['Chest X-ray PA and Lateral', 'Chest X-ray Portable', 'CT Chest without contrast']

# Lung field findings
lung_findings = ['Clear', 'Infiltrate present', 'Consolidation', 'Nodule', 'Mass']

# Heart size categories
heart_sizes = ['Normal', 'Borderline enlarged', 'Enlarged']

# Pleural findings
pleural_findings = ['No effusion', 'Small effusion', 'Moderate effusion', 'Large effusion']

# Bone findings
bone_findings = ['No acute fracture', 'Rib fracture', 'Clavicle fracture', 'Degenerative changes']

# Overall impression categories
impression_categories = ['Normal', 'Abnormal - minor', 'Abnormal - significant', 'Critical']

In [ ]:
# Generate synthetic patient examination records
num_samples = 150  # Number of examination records to generate

# Create empty lists to store each data field
data = {
    'patient_id': [],
    'age': [],
    'gender': [],
    'exam_date': [],
    'indication': [],
    'modality': [],
    'lung_finding': [],
    'heart_size': [],
    'pleural_finding': [],
    'bone_finding': [],
    'additional_finding': [],
    'impression_category': []
}

# Generate each examination record
for i in range(num_samples):
    # Generate unique patient ID
    data['patient_id'].append(f"MRN{100000 + i}")

    # Generate random age between 18 and 90
    data['age'].append(np.random.randint(18, 91))

    # Randomly select gender
    data['gender'].append(np.random.choice(genders))

    # Generate random exam date within the last year
    days_ago = np.random.randint(0, 365)
    exam_date = datetime.now() - timedelta(days=days_ago)
    data['exam_date'].append(exam_date.strftime('%Y-%m-%d'))

    # Randomly select clinical parameters
    data['indication'].append(np.random.choice(indications))
    data['modality'].append(np.random.choice(modalities))
    data['lung_finding'].append(np.random.choice(lung_findings))
    data['heart_size'].append(np.random.choice(heart_sizes))
    data['pleural_finding'].append(np.random.choice(pleural_findings))
    data['bone_finding'].append(np.random.choice(bone_findings))

    # Add occasional additional findings (30% chance)
    additional = np.random.choice(
        ['None', 'Calcified granuloma', 'Surgical clips', 'Medical device present'],
        p=[0.7, 0.1, 0.1, 0.1]
    )
    data['additional_finding'].append(additional)

    # Determine impression category based on findings
    # Critical findings: large effusion, mass, or severe consolidation
    if 'Mass' in data['lung_finding'][i] or 'Large effusion' in data['pleural_finding'][i]:
        data['impression_category'].append('Critical')
    # Significant findings: consolidation, moderate effusion, or fracture
    elif any(x in data['lung_finding'][i] for x in ['Consolidation', 'Infiltrate']) or \
         'Moderate effusion' in data['pleural_finding'][i] or 'fracture' in data['bone_finding'][i].lower():
        data['impression_category'].append('Abnormal - significant')
    # Minor findings: small effusion, nodule, enlarged heart
    elif 'Small effusion' in data['pleural_finding'][i] or \
         'Nodule' in data['lung_finding'][i] or \
         'enlarged' in data['heart_size'][i].lower():
        data['impression_category'].append('Abnormal - minor')
    # Normal findings
    else:
        data['impression_category'].append('Normal')

# Create DataFrame from the generated data
df_radiology = pd.DataFrame(data)

print(f"Generated {len(df_radiology)} synthetic radiology examination records")

## Step 2: Load and explore the dataset

Let's examine the structure and characteristics of your radiology dataset to understand the information available for report generation.

In [ ]:
# Display the first few records to understand data structure
print("First 5 examination records:")
print(df_radiology.head())

In [ ]:
# Display dataset information including data types and non-null counts
print("Dataset Information:")
print(df_radiology.info())

In [ ]:
# Display statistical summary for numerical columns
print("Statistical Summary:")
print(df_radiology.describe())

In [ ]:
# Check for missing values in the dataset
print("Missing values per column:")
print(df_radiology.isnull().sum())

In [ ]:
# Display distribution of impression categories
print("Distribution of Impression Categories:")
print(df_radiology['impression_category'].value_counts())
print("\nPercentage distribution:")
print(df_radiology['impression_category'].value_counts(normalize=True) * 100)

## Step 3: Visualize dataset characteristics

Creating visualizations helps us understand the distribution of findings and patient demographics in your dataset.

In [ ]:
# Create a figure with multiple subplots for various distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Age distribution
axes[0, 0].hist(df_radiology['age'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Age (years)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Patient Ages')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Gender distribution
gender_counts = df_radiology['gender'].value_counts()
axes[0, 1].bar(gender_counts.index, gender_counts.values, color=['lightcoral', 'lightblue'])
axes[0, 1].set_xlabel('Gender')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Distribution by Gender')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Impression category distribution
impression_counts = df_radiology['impression_category'].value_counts()
axes[1, 0].barh(impression_counts.index, impression_counts.values, color='lightgreen')
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_ylabel('Impression Category')
axes[1, 0].set_title('Distribution of Impression Categories')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Lung findings distribution
lung_counts = df_radiology['lung_finding'].value_counts()
axes[1, 1].barh(lung_counts.index, lung_counts.values, color='lightsalmon')
axes[1, 1].set_xlabel('Count')
axes[1, 1].set_ylabel('Lung Finding')
axes[1, 1].set_title('Distribution of Lung Findings')
axes[1, 1].grid(True, alpha=0.3)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# Create a heatmap showing correlation between categorical findings
# First, encode categorical variables as numeric
from sklearn.preprocessing import LabelEncoder

# Create a copy for encoding
df_encoded = df_radiology.copy()

# Encode categorical columns to numeric values
categorical_cols = ['lung_finding', 'heart_size', 'pleural_finding', 'bone_finding', 'impression_category']
le = LabelEncoder()

for col in categorical_cols:
    df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col])

# Calculate correlation matrix for encoded variables
encoded_cols = [col + '_encoded' for col in categorical_cols]
correlation_matrix = df_encoded[encoded_cols].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            xticklabels=categorical_cols,
            yticklabels=categorical_cols,
            center=0)
plt.title('Correlation Between Clinical Findings')
plt.tight_layout()
plt.show()

## Step 4: Build report generation templates

You'll create template-based functions to generate structured radiology reports. Each report section (Clinical History, Technique, Findings, Impression) will have its own generation logic.

In [ ]:
def generate_clinical_history(row):
    """
    Generate the Clinical History section of the report.

    Parameters:
    row: A pandas Series containing patient and examination data

    Returns:
    str: Formatted clinical history text
    """
    # Extract patient age and gender
    age = row['age']
    gender = row['gender'].lower()

    # Extract clinical indication
    indication = row['indication']

    # Create clinical history statement
    history = f"{age}-year-old {gender} with {indication.lower()}."

    return history

In [ ]:
def generate_technique(row):
    """
    Generate the Technique section describing the imaging method.

    Parameters:
    row: A pandas Series containing examination data

    Returns:
    str: Formatted technique description
    """
    # Extract imaging modality
    modality = row['modality']

    # Extract examination date
    exam_date = row['exam_date']

    # Create technique description
    technique = f"{modality} performed on {exam_date}."

    # Add comparison statement (randomly for variety)
    if np.random.random() > 0.5:
        technique += " No prior studies available for comparison."
    else:
        technique += " Comparison made with prior study when available."

    return technique

In [ ]:
def generate_findings(row):
    """
    Generate detailed Findings section based on anatomical observations.

    Parameters:
    row: A pandas Series containing examination findings

    Returns:
    str: Formatted findings text with all observations
    """
    findings = []

    # Describe lung findings
    lung_finding = row['lung_finding']
    if lung_finding == 'Clear':
        findings.append("The lungs are clear bilaterally without focal consolidation, effusion, or pneumothorax.")
    elif lung_finding == 'Infiltrate present':
        # Randomly assign location
        location = np.random.choice(['right lower lobe', 'left lower lobe', 'right upper lobe', 'left upper lobe'])
        findings.append(f"Patchy infiltrate noted in the {location}, suggestive of infection or inflammation.")
    elif lung_finding == 'Consolidation':
        location = np.random.choice(['right lower lobe', 'left lower lobe', 'bilateral lower lobes'])
        findings.append(f"Consolidation present in the {location}, consistent with pneumonia.")
    elif lung_finding == 'Nodule':
        size = np.random.choice(['5mm', '8mm', '12mm'])
        location = np.random.choice(['right upper lobe', 'left upper lobe', 'right middle lobe'])
        findings.append(f"A {size} nodule is identified in the {location}. Further evaluation with CT recommended.")
    elif lung_finding == 'Mass':
        size = np.random.choice(['2.5cm', '3.2cm', '4.0cm'])
        location = np.random.choice(['right upper lobe', 'left upper lobe', 'right lower lobe'])
        findings.append(f"A {size} mass is present in the {location}. Urgent further evaluation recommended.")

    # Describe heart size
    heart_size = row['heart_size']
    if heart_size == 'Normal':
        findings.append("The cardiac silhouette is normal in size.")
    elif heart_size == 'Borderline enlarged':
        findings.append("The cardiac silhouette is borderline enlarged. Clinical correlation suggested.")
    else:
        findings.append("Cardiomegaly is present. Recommend clinical correlation and possible echocardiography.")

    # Describe mediastinum
    findings.append("The mediastinal contours are within normal limits.")

    # Describe pleural findings
    pleural_finding = row['pleural_finding']
    if pleural_finding == 'No effusion':
        findings.append("No pleural effusion or pneumothorax.")
    elif 'effusion' in pleural_finding:
        severity = pleural_finding.split()[0]  # Extract 'Small', 'Moderate', or 'Large'
        side = np.random.choice(['right', 'left', 'bilateral'])
        findings.append(f"{severity} {side} pleural effusion noted.")

    # Describe bone findings
    bone_finding = row['bone_finding']
    if bone_finding == 'No acute fracture':
        findings.append("The visualized osseous structures show no acute fracture.")
    elif 'fracture' in bone_finding.lower():
        findings.append(f"{bone_finding} identified.")
    else:
        findings.append(f"The osseous structures demonstrate {bone_finding.lower()}.")

    # Add additional findings if present
    additional = row['additional_finding']
    if additional != 'None':
        findings.append(f"{additional} noted.")

    # Join all findings with spaces
    return " ".join(findings)

In [ ]:
def generate_impression(row):
    """
    Generate the Impression section with summary and recommendations.

    Parameters:
    row: A pandas Series containing examination findings

    Returns:
    str: Formatted impression with numbered conclusions
    """
    impressions = []
    impression_count = 1

    # Summarize lung findings
    lung_finding = row['lung_finding']
    if lung_finding == 'Clear':
        impressions.append(f"{impression_count}. No acute cardiopulmonary abnormality.")
        impression_count += 1
    elif lung_finding == 'Infiltrate present':
        impressions.append(f"{impression_count}. Pulmonary infiltrate, likely infectious or inflammatory. Clinical correlation recommended.")
        impression_count += 1
    elif lung_finding == 'Consolidation':
        impressions.append(f"{impression_count}. Pneumonia. Recommend appropriate antibiotic therapy and follow-up imaging.")
        impression_count += 1
    elif lung_finding == 'Nodule':
        impressions.append(f"{impression_count}. Pulmonary nodule. Recommend CT chest for further characterization and follow-up per Fleischner criteria.")
        impression_count += 1
    elif lung_finding == 'Mass':
        impressions.append(f"{impression_count}. Lung mass. URGENT: Recommend immediate CT chest and oncology consultation.")
        impression_count += 1

    # Summarize cardiac findings
    if row['heart_size'] != 'Normal':
        if row['heart_size'] == 'Enlarged':
            impressions.append(f"{impression_count}. Cardiomegaly. Suggest echocardiography for further evaluation.")
        else:
            impressions.append(f"{impression_count}. Borderline cardiomegaly.")
        impression_count += 1

    # Summarize pleural findings
    if row['pleural_finding'] != 'No effusion':
        severity = row['pleural_finding'].split()[0]
        if severity == 'Large':
            impressions.append(f"{impression_count}. Large pleural effusion. Consider thoracentesis for diagnostic and therapeutic purposes.")
        else:
            impressions.append(f"{impression_count}. {severity} pleural effusion.")
        impression_count += 1

    # Summarize bone findings
    if 'fracture' in row['bone_finding'].lower():
        impressions.append(f"{impression_count}. {row['bone_finding']}. Orthopedic consultation may be warranted.")
        impression_count += 1

    # If no significant findings, ensure there's at least one impression
    if len(impressions) == 0:
        impressions.append("1. No acute cardiopulmonary abnormality.")

    # Join impressions with newlines
    return "\n".join(impressions)

## Step 5: Generate complete radiology reports

Now you'll combine all sections to create complete, formatted radiology reports for each examination.

In [ ]:
def generate_complete_report(row):
    """
    Generate a complete radiology report with all standard sections.

    Parameters:
    row: A pandas Series containing all examination data

    Returns:
    str: Complete formatted radiology report
    """
    # Create report header
    report = "="*80 + "\n"
    report += "RADIOLOGY REPORT\n"
    report += "="*80 + "\n\n"

    # Add patient information
    report += f"Patient ID: {row['patient_id']}\n"
    report += f"Age: {row['age']} years\n"
    report += f"Gender: {row['gender']}\n"
    report += f"Exam Date: {row['exam_date']}\n"
    report += "\n" + "-"*80 + "\n\n"

    # Add Clinical History section
    report += "CLINICAL HISTORY:\n"
    report += generate_clinical_history(row) + "\n\n"

    # Add Technique section
    report += "TECHNIQUE:\n"
    report += generate_technique(row) + "\n\n"

    # Add Findings section
    report += "FINDINGS:\n"
    report += generate_findings(row) + "\n\n"

    # Add Impression section
    report += "IMPRESSION:\n"
    report += generate_impression(row) + "\n"

    # Add report footer
    report += "\n" + "="*80 + "\n"

    return report

In [ ]:
# Generate reports for all examinations in the dataset
print("Generating radiology reports...\n")

# Create a new column to store generated reports
df_radiology['generated_report'] = df_radiology.apply(generate_complete_report, axis=1)

print(f"Successfully generated {len(df_radiology)} reports!")

In [ ]:
# Display example reports for different impression categories
print("EXAMPLE REPORT 1: Normal Study")
print("="*80)

# Find and display a normal study
normal_reports = df_radiology[df_radiology['impression_category'] == 'Normal']
if len(normal_reports) > 0:
    print(normal_reports.iloc[0]['generated_report'])
else:
    print("No normal studies found in this dataset.")

In [ ]:
print("\nEXAMPLE REPORT 2: Abnormal - Significant Finding")
print("="*80)

# Find and display a significant abnormal study
abnormal_reports = df_radiology[df_radiology['impression_category'] == 'Abnormal - significant']
if len(abnormal_reports) > 0:
    print(abnormal_reports.iloc[0]['generated_report'])
else:
    print("No significant abnormal studies found in this dataset.")

In [ ]:
print("\nEXAMPLE REPORT 3: Critical Finding")
print("="*80)

# Find and display a critical study if available
critical_reports = df_radiology[df_radiology['impression_category'] == 'Critical']
if len(critical_reports) > 0:
    print(critical_reports.iloc[0]['generated_report'])
else:
    print("No critical findings in this dataset sample.")

## Step 6: Analyze report characteristics

Let's analyze the characteristics of your generated reports to ensure quality and consistency.

In [ ]:
# Calculate report length statistics
df_radiology['report_length'] = df_radiology['generated_report'].apply(len)
df_radiology['report_word_count'] = df_radiology['generated_report'].apply(lambda x: len(x.split()))

# Display statistics
print("Report Length Statistics:")
print(f"Average characters: {df_radiology['report_length'].mean():.0f}")
print(f"Average words: {df_radiology['report_word_count'].mean():.0f}")
print(f"Min words: {df_radiology['report_word_count'].min()}")
print(f"Max words: {df_radiology['report_word_count'].max()}")

In [ ]:
# Analyze report length by impression category
report_length_by_category = df_radiology.groupby('impression_category')['report_word_count'].agg(['mean', 'min', 'max'])

print("\nReport Word Count by Impression Category:")
print(report_length_by_category)

In [ ]:
# Visualize report length distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Overall word count distribution
axes[0].hist(df_radiology['report_word_count'], bins=20, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Report Lengths')
axes[0].axvline(df_radiology['report_word_count'].mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Word count by impression category
impression_order = ['Normal', 'Abnormal - minor', 'Abnormal - significant', 'Critical']
df_radiology_sorted = df_radiology.copy()
df_radiology_sorted['impression_category'] = pd.Categorical(
    df_radiology_sorted['impression_category'],
    categories=impression_order,
    ordered=True
)

# Create box plot
df_radiology_sorted.boxplot(column='report_word_count', by='impression_category', ax=axes[1])
axes[1].set_xlabel('Impression Category')
axes[1].set_ylabel('Word Count')
axes[1].set_title('Report Length by Impression Category')
plt.suptitle('')  # Remove automatic title
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Create report quality metrics

You'll implement basic quality checks to ensure reports contain all required sections and appropriate content.

In [ ]:
def check_report_completeness(report_text):
    """
    Check if a report contains all required sections.

    Parameters:
    report_text: String containing the complete report

    Returns:
    dict: Dictionary with boolean values for each required section
    """
    # Define required sections
    required_sections = {
        'has_clinical_history': 'CLINICAL HISTORY:' in report_text,
        'has_technique': 'TECHNIQUE:' in report_text,
        'has_findings': 'FINDINGS:' in report_text,
        'has_impression': 'IMPRESSION:' in report_text,
        'has_patient_id': 'Patient ID:' in report_text
    }

    # Check if report is complete (all sections present)
    required_sections['is_complete'] = all(required_sections.values())

    return required_sections

In [ ]:
# Apply completeness check to all reports
completeness_results = df_radiology['generated_report'].apply(check_report_completeness)

# Convert results to DataFrame for easier analysis
completeness_df = pd.DataFrame(completeness_results.tolist())

# Display completeness statistics
print("Report Completeness Analysis:")
print("="*50)
for col in completeness_df.columns:
    count = completeness_df[col].sum()
    percentage = (count / len(completeness_df)) * 100
    print(f"{col}: {count}/{len(completeness_df)} ({percentage:.1f}%)")

In [ ]:
def analyze_report_content(report_text):
    """
    Analyze content characteristics of a report.

    Parameters:
    report_text: String containing the complete report

    Returns:
    dict: Dictionary with various content metrics
    """
    # Count specific medical terms
    metrics = {
        'mentions_recommendation': 'recommend' in report_text.lower(),
        'mentions_urgent': 'urgent' in report_text.lower(),
        'mentions_normal': 'normal' in report_text.lower(),
        'mentions_abnormal': any(word in report_text.lower() for word in ['abnormal', 'consolidation', 'effusion', 'mass', 'fracture']),
        'numbered_impression': any(str(i) + '.' in report_text for i in range(1, 10))
    }

    return metrics

In [ ]:
# Analyze content of all reports
content_results = df_radiology['generated_report'].apply(analyze_report_content)

# Convert to DataFrame
content_df = pd.DataFrame(content_results.tolist())

# Display content analysis
print("Report Content Analysis:")
print("="*50)
for col in content_df.columns:
    count = content_df[col].sum()
    percentage = (count / len(content_df)) * 100
    print(f"{col}: {count}/{len(content_df)} ({percentage:.1f}%)")

## Step 8: Save generated reports

Let's save your generated reports to files for future use and review.

In [ ]:
# Save summary dataset with key information
summary_columns = [
    'patient_id', 'age', 'gender', 'exam_date', 'indication',
    'lung_finding', 'heart_size', 'pleural_finding', 'bone_finding',
    'impression_category', 'report_word_count'
]

df_summary = df_radiology[summary_columns]
df_summary.to_csv('radiology_reports_summary.csv', index=False)
print("Saved report summary to: radiology_reports_summary.csv")

In [ ]:
# Save first 10 complete reports to individual text files
import os

# Create directory for reports if it doesn't exist
reports_dir = 'generated_reports'
if not os.path.exists(reports_dir):
    os.makedirs(reports_dir)

# Save first 10 reports
num_reports_to_save = min(10, len(df_radiology))
for i in range(num_reports_to_save):
    # Get patient ID and report
    patient_id = df_radiology.iloc[i]['patient_id']
    report = df_radiology.iloc[i]['generated_report']

    # Create filename
    filename = f"{reports_dir}/report_{patient_id}.txt"

    # Write report to file
    with open(filename, 'w') as f:
        f.write(report)

print(f"Saved {num_reports_to_save} individual reports to: {reports_dir}/")

## Exercises

Now it's your turn to extend and improve the report generation system!

### Exercise 1: Add recommendations section

Create a new function that generates a "RECOMMENDATIONS" section based on the impression category. For example:
- Normal: "Routine follow-up as clinically indicated."
- Critical findings: "Immediate clinical correlation and urgent follow-up required."
- Nodules: "Follow-up CT in 3-6 months per Fleischner criteria."

In [ ]:
# Your code here
def generate_recommendations(row):
    """
    Generate recommendations based on findings.

    Parameters:
    row: A pandas Series containing examination findings

    Returns:
    str: Formatted recommendations
    """
    # TODO: Implement recommendation logic
    pass

<details>
<summary>Hint</summary>

Use the impression_category and specific findings (lung_finding, pleural_finding) to determine appropriate recommendations. Consider using if-elif-else statements to handle different scenarios. Look at the generate_impression function for inspiration on how to structure your logic.

</details>

<details>
<summary>Solution</summary>

```python
def generate_recommendations(row):
    """
    Generate recommendations based on findings.
    
    Parameters:
    row: A pandas Series containing examination findings
    
    Returns:
    str: Formatted recommendations
    """
    recommendations = []
    
    # Check impression category for urgent findings
    if row['impression_category'] == 'Critical':
        recommendations.append("URGENT: Immediate clinical correlation required.")
        recommendations.append("Contact referring physician immediately.")
    
    # Specific recommendations based on findings
    if row['lung_finding'] == 'Mass':
        recommendations.append("Recommend CT chest with contrast for further characterization.")
        recommendations.append("Consider oncology consultation.")
    elif row['lung_finding'] == 'Nodule':
        recommendations.append("Follow-up CT in 3-6 months per Fleischner criteria.")
    elif row['lung_finding'] == 'Consolidation':
        recommendations.append("Recommend appropriate antibiotic therapy.")
        recommendations.append("Follow-up chest X-ray in 4-6 weeks to document resolution.")
    
    # Cardiac recommendations
    if row['heart_size'] == 'Enlarged':
        recommendations.append("Recommend echocardiography for cardiac evaluation.")
    
    # Pleural effusion recommendations
    if 'Large effusion' in row['pleural_finding']:
        recommendations.append("Consider thoracentesis for diagnostic and therapeutic purposes.")
    
    # Fracture recommendations
    if 'fracture' in row['bone_finding'].lower():
        recommendations.append("Orthopedic consultation may be warranted.")
    
    # Default recommendation for normal studies
    if len(recommendations) == 0:
        recommendations.append("Routine clinical follow-up as indicated.")
    
    # Join recommendations
    return "\n".join([f"- {rec}" for rec in recommendations])

# Test the function on a sample
print("RECOMMENDATIONS:")
print(generate_recommendations(df_radiology.iloc[0]))
```

</details>

### Exercise 2: Create report summary statistics

Write a function that generates statistical insights about all reports, including:
- Total number of reports generated
- Percentage of normal vs abnormal studies
- Most common findings
- Average report length by impression category

In [ ]:
# Your code here
def generate_report_statistics(df):
    """
    Generate comprehensive statistics about the report dataset.

    Parameters:
    df: DataFrame containing radiology reports

    Returns:
    str: Formatted statistics summary
    """
    # TODO: Implement statistics generation
    pass

<details>
<summary>Hint</summary>

Use pandas methods like value_counts(), groupby(), and describe() to calculate statistics. You can calculate percentages using value_counts(normalize=True). Format the output as a string with clear section headers.

</details>

<details>
<summary>Solution</summary>

```python
def generate_report_statistics(df):
    """
    Generate comprehensive statistics about the report dataset.
    
    Parameters:
    df: DataFrame containing radiology reports
    
    Returns:
    str: Formatted statistics summary
    """
    stats = []
    stats.append("="*60)
    stats.append("RADIOLOGY REPORT STATISTICS")
    stats.append("="*60)
    
    # Total reports
    stats.append(f"\nTotal Reports Generated: {len(df)}")
    
    # Normal vs Abnormal
    stats.append("\n" + "-"*60)
    stats.append("Impression Category Distribution:")
    stats.append("-"*60)
    impression_dist = df['impression_category'].value_counts()
    impression_pct = df['impression_category'].value_counts(normalize=True) * 100
    for category in impression_dist.index:
        stats.append(f"{category}: {impression_dist[category]} ({impression_pct[category]:.1f}%)")
    
    # Most common findings
    stats.append("\n" + "-"*60)
    stats.append("Most Common Findings:")
    stats.append("-"*60)
    stats.append(f"Lung: {df['lung_finding'].mode()[0]} ({df['lung_finding'].value_counts().iloc[0]} cases)")
    stats.append(f"Heart: {df['heart_size'].mode()[0]} ({df['heart_size'].value_counts().iloc[0]} cases)")
    stats.append(f"Pleura: {df['pleural_finding'].mode()[0]} ({df['pleural_finding'].value_counts().iloc[0]} cases)")
    
    # Report length statistics
    stats.append("\n" + "-"*60)
    stats.append("Report Length Statistics (words):")
    stats.append("-"*60)
    stats.append(f"Average: {df['report_word_count'].mean():.1f}")
    stats.append(f"Median: {df['report_word_count'].median():.1f}")
    stats.append(f"Range: {df['report_word_count'].min()} - {df['report_word_count'].max()}")
    
    # Length by category
    stats.append("\n" + "-"*60)
    stats.append("Average Report Length by Category:")
    stats.append("-"*60)
    avg_length = df.groupby('impression_category')['report_word_count'].mean()
    for category, length in avg_length.items():
        stats.append(f"{category}: {length:.1f} words")
    
    stats.append("\n" + "="*60)
    
    return "\n".join(stats)

# Generate and display statistics
print(generate_report_statistics(df_radiology))
```

</details>

### Exercise 3: Implement report comparison function

Create a function that compares two radiology reports for the same patient (e.g., current vs prior study) and identifies changes in findings. This is useful for follow-up examinations.

In [ ]:
# Your code here
def compare_reports(prior_row, current_row):
    """
    Compare two reports and identify changes.

    Parameters:
    prior_row: Series containing prior examination data
    current_row: Series containing current examination data

    Returns:
    str: Formatted comparison summary
    """
    # TODO: Implement comparison logic
    pass

<details>
<summary>Hint</summary>

Compare key findings fields (lung_finding, heart_size, pleural_finding, bone_finding) between the two reports. Identify what changed, what improved, what worsened, and what remained stable. Use descriptive language like "improved", "worsened", "unchanged", "new finding", or "resolved".

</details>

<details>
<summary>Solution</summary>

```python
def compare_reports(prior_row, current_row):
    """
    Compare two reports and identify changes.
    
    Parameters:
    prior_row: Series containing prior examination data
    current_row: Series containing current examination data
    
    Returns:
    str: Formatted comparison summary
    """
    comparison = []
    comparison.append("="*70)
    comparison.append("COMPARISON WITH PRIOR STUDY")
    comparison.append("="*70)
    comparison.append(f"Prior exam: {prior_row['exam_date']}")
    comparison.append(f"Current exam: {current_row['exam_date']}")
    comparison.append("\n" + "-"*70)
    
    changes = []
    
    # Compare lung findings
    if prior_row['lung_finding'] != current_row['lung_finding']:
        changes.append(f"Lung findings: {prior_row['lung_finding']} → {current_row['lung_finding']}")
        # Determine if improvement or worsening
        if current_row['lung_finding'] == 'Clear':
            changes.append("  Status: IMPROVED (resolved)")
        elif prior_row['lung_finding'] == 'Clear':
            changes.append("  Status: NEW FINDING")
        else:
            changes.append("  Status: CHANGED")
    
    # Compare heart size
    if prior_row['heart_size'] != current_row['heart_size']:
        changes.append(f"Heart size: {prior_row['heart_size']} → {current_row['heart_size']}")
        if current_row['heart_size'] == 'Normal':
            changes.append("  Status: IMPROVED")
        elif prior_row['heart_size'] == 'Normal':
            changes.append("  Status: WORSENED")
    
    # Compare pleural findings
    if prior_row['pleural_finding'] != current_row['pleural_finding']:
        changes.append(f"Pleural findings: {prior_row['pleural_finding']} → {current_row['pleural_finding']}")
        if current_row['pleural_finding'] == 'No effusion':
            changes.append("  Status: IMPROVED (resolved)")
        elif prior_row['pleural_finding'] == 'No effusion':
            changes.append("  Status: NEW FINDING")
        elif 'Small' in current_row['pleural_finding'] and 'Large' in prior_row['pleural_finding']:
            changes.append("  Status: IMPROVED (decreased)")
        else:
            changes.append("  Status: CHANGED")
    
    # Compare bone findings
    if prior_row['bone_finding'] != current_row['bone_finding']:
        changes.append(f"Bone findings: {prior_row['bone_finding']} → {current_row['bone_finding']}")
    
    # Add changes to comparison
    if len(changes) > 0:
        comparison.append("\nCHANGES IDENTIFIED:")
        comparison.extend(changes)
    else:
        comparison.append("\nNo significant interval change.")
    
    # Overall impression comparison
    comparison.append("\n" + "-"*70)
    comparison.append(f"Prior impression: {prior_row['impression_category']}")
    comparison.append(f"Current impression: {current_row['impression_category']}")
    
    comparison.append("="*70)
    
    return "\n".join(comparison)

# Test with two different reports
if len(df_radiology) >= 2:
    print(compare_reports(df_radiology.iloc[0], df_radiology.iloc[1]))
```

</details>

## Congratulations

You have successfully completed this lab on **Medical Report Generation System**. You learned how to generate synthetic medical data, design template-based clinical reports, and ensure quality and consistency in automated text generation. These skills form the foundation for developing reliable AI-assisted reporting systems used in modern healthcare.

## Authors
Ramesh Sannareddy

Copyright © 2025 SkillUp. All rights reserved.